# Week 3 Pipeline: UTM FAI Student FAQ Assistant (Pathway A: PDF/TXT)

**Selected sources (both uploaded to `/content/week3_project/data/raw/`):**

- **S01:** `FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf` (UTM Academic Regulations for Full-Time Undergraduate Studies Programmes, 2026)
- **S02:** `Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf` (FAI programme page: Bachelor in Artificial Intelligence with Honours)

**Topic 2 requirements this data supports:** FR1, FR4 (answers from the regulations, including multi-part answers), FR3 / RAI1 (source reference and verifiable source identity), RAI3 (source version), NFR4 (reproducible, updatable source handling).

**Pathway:** A (PDF/TXT) only. Pathways B to F are alternatives and are not run, because this project has no tabular, image, JSON, audio or time-series data.

**Cell order:** Cell 1 → upload both sources → Cell 3 → Cell 4 → Cell 4A → for each source (S01 then S02): Step A3 inspection → Cell 5A1 → B3 comparison → Cell 5A2 → B3 validation → B4.1 lineage → Cell 6A → Cell 6B → Cell 7 → Cell 8A (once per source) → Cell 9. (Cell 2 is optional and skipped: no dummy ZIP is used.)

## Part A: Start with the Data

### Step A2: Set up the Colab workspace
**Cell 1 (REQUIRED).** Run first, and again after any runtime reset.

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


Upload `FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf` and `Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf` into `/content/week3_project/data/raw/` using the Colab Files panel, then run Cell 3.

**Cell 3 (REQUIRED).** Confirm the uploaded input exists and is accessible.

In [2]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf bytes: 1372769
FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf bytes: 779259
Upload verified. Select your relevant input(s) for Step A3.


### Common metadata helpers (REQUIRED for all pathways)
**Cell 4:** `select_source()` for unambiguous input selection, `source_hash()` for SHA-256 checksums, `save_manifest()` for lineage.

In [3]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


**Cell 4A (REQUIRED).** Initialise the common lineage registry.

In [4]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


### Source S01 (UTM Academic Regulations for Full-Time Undergraduate Studies Programmes, 2026): Step A3, inspect the selected data source (REQUIRED)
Selected source path and type: structured? No. **Unstructured: text (PDF)**. Two representative subsets of this PDF are inspected: a body-text page and a second, differently laid-out page (a table page where one exists). Observed limitations feed the Part B decisions.

In [5]:
# Step A3 — Inspect the selected source (REQUIRED). Added for the lab's inspection evidence.
from pypdf import PdfReader
import re, statistics
from collections import Counter

SOURCE_NAME = 'FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf'
SRC = select_source(relative_path=SOURCE_NAME)
reader = PdfReader(str(SRC))
page_text = [(p.extract_text() or "") for p in reader.pages]
n_pages = len(page_text)
chars = [len(t) for t in page_text]
empty_pages = [i + 1 for i, c in enumerate(chars) if c < 20]

print("Source path:", SRC.relative_to(RAW_DIR))
print("File size (bytes):", SRC.stat().st_size)
print("Pages:", n_pages)
print("PDF metadata title:", (reader.metadata.title if reader.metadata else None))
print("Characters per page: min/median/max =", min(chars), "/", int(statistics.median(chars)), "/", max(chars))
print("Pages with (almost) no extractable text:", empty_pages)

# Subset 1: first page that looks like body text (>= 800 characters)
body_idx = next((i for i, c in enumerate(chars) if c >= 800), 0)
# Subset 2: first page whose text contains a table caption such as "Table 5"
table_idx = next((i for i, t in enumerate(page_text) if re.search(r"^Table \d+", t, re.M)), None)
sub2_label = "Subset 2 (page with a table caption)"
if table_idx is None:
    # No "Table N" caption in this source: use the page with the most lines (list/table-like layout)
    table_idx = max((i for i in range(n_pages) if i != body_idx), key=lambda i: page_text[i].count("\n"))
    sub2_label = "Subset 2 (page with the most lines; no table caption in this source)"

for label, idx in [("Subset 1 (body text page)", body_idx), (sub2_label, table_idx)]:
    if idx is None:
        print("\n" + label + ": none found")
        continue
    t = page_text[idx]
    print("\n" + label + ": PDF page", idx + 1, "| characters:", len(t), "| lines:", t.count("\n") + 1)
    print(repr(t[:400]))

# Layout checks that affect chunking
norm = Counter()
for t in page_text:
    for line in set(l.strip() for l in t.splitlines()):
        if 0 < len(line) < 100:
            norm[re.sub(r"\d+", "#", line)] += 1
repeated = [(l, c) for l, c in norm.most_common(8) if c >= max(3, n_pages // 4)]
hyphen_breaks = sum(len(re.findall(r"\w-\n\w", t)) for t in page_text)
table_captions = sum(len(re.findall(r"^Table \d+", t, re.M)) for t in page_text)
article_headings = sum(len(re.findall(r"^Article \d+", t, re.M)) for t in page_text)
blank_line_breaks = sum(len(re.findall(r"\n\s*\n", t)) for t in page_text)
print("\nRepeated short lines (digits shown as #), appears on N pages:", repeated)
print("Hyphenated line breaks:", hyphen_breaks)
print("Table captions found:", table_captions)
print("'Article N' headings found at line start:", article_headings)
print("Blank-line paragraph breaks across the whole document:", blank_line_breaks)

# Source identity lines (edition / ISBN / publisher) from the first pages
ident = [l.strip() for t in page_text[:4] for l in t.splitlines() if re.search(r"Edition|ISBN|Penerbit|Published", l, re.I)]
print("\nSource identity lines (first pages):", ident[:6])

# Evidence of how a table is extracted
tbl_excerpt = []
if table_idx is not None:
    _lines = page_text[table_idx].splitlines()
    for _j, _l in enumerate(_lines):
        if re.match(r"Table \d+", _l):
            tbl_excerpt = _lines[_j:_j + 8]
            break
    if not tbl_excerpt:
        tbl_excerpt = _lines[:8]   # no table caption: show the first lines of the subset-2 page
print("Table excerpt as extracted:", tbl_excerpt)

INSPECTION = {
    "identity_lines": ident[:6], "table_excerpt": tbl_excerpt,
    "pages": n_pages, "file_bytes": SRC.stat().st_size, "chars_total": sum(chars),
    "chars_min": min(chars), "chars_median": int(statistics.median(chars)), "chars_max": max(chars),
    "empty_pages": empty_pages, "subset1_page": body_idx + 1,
    "subset2_page": (table_idx + 1) if table_idx is not None else None,
    "repeated_lines": repeated, "hyphen_breaks": hyphen_breaks, "table_captions": table_captions,
    "article_headings": article_headings, "blank_line_breaks": blank_line_breaks,
}
assert sum(chars) > 0, "No extractable text: this source may need OCR."

Source path: FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf
File size (bytes): 779259
Pages: 81
PDF metadata title: None
Characters per page: min/median/max = 0 / 1553 / 2402
Pages with (almost) no extractable text: [1, 2, 3, 11, 13, 15, 81]

Subset 1 (body text page): PDF page 5 | characters: 1647 | lines: 45
' \nFirst Edition 2026 \n© MOHD SHAFRY MOHD RAHIM & INTAN ZAURAH MAT DARUS 2026 \n \nHak cipta terpelihara. Tiada dibenarkan mengeluar ulang mana-mana bahagian artikel, ilustrasi, dan \nisi kandungan buku ini dalam apa juga bentuk dan cara apa jua sama ada dengan cara elektronik, \nfotokopi, mekanikal, atau cara lain sebelum mendapat izin bertulis daripada Timbalan Naib Canselor \n(Penyelidikan & Inovasi)'

Subset 2 (page with a table caption): PDF page 26 | characters: 1027 | lines: 32
'9\nAcademic Regulations Section 2\nTable 1  Academic Session *\n* 1 week of  mid-semester break for Semester I and II is allocated subject to the academic \nsession calendar set by the University .\

## Part B: Prepare the Data

### Step B1/B2: Pathway A (PDF/TXT) — parsing, cleaning and chunking
**Cell 5A1 (CONDITIONAL REQUIRED), source S01.** Extract text, clean, and compare fixed-size and paragraph-aware chunking.

In [6]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Project source S01: UTM Academic Regulations for Full-Time Undergraduate Studies Programmes, 2026
f = select_source(
    relative_path='FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf'
)

# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf
Pages/segments: 81
Extracted characters: 105212
Cleaned characters: 104682
Content sample: 'www.penerbit.utm.my\n2026\n/gid00006/gid00031/gid00036/gid00047/gid00042/gid00045/gid00046\n/gid00014/gid00042/gid00035/gid00031/gid00001/gid00020/gid00035/gid00028/gid00033/gid00045/gid00052/gid00001/gid00014/gid00042/gid00035/gid00031/gid00001/gid00019/gid00028/gid00035/gid00036/gid00040\n/gid00010/gi'

--- Chunking Comparison ---
Fixed chunks: 210
Paragraph-aware chunks: 213

Fixed sample:
'www.penerbit.utm.my\n2026\n/gid00006/gid00031/gid00036/gid00047/gid00042/gid00045/gid00046\n/gid00014/gid00042/gid00035/gid00031/gid00001/gid00020/gid00035/gid00028/gid00033/gid00045/'

Paragraph-aware sample:
'www.penerbit.utm.my\n2026\n/gid00006/gid00031/gid00036/gid00047/gid00042/gid00045/gid00046\n/gid00014/gid00042/gid00035/gid00031/gid00001/gid00020/gid00035/gid00028/gid00033/gid00045/'

PATHWAY A1 completed successfully.


**Step B3 (part 1): compare the two approaches on the same cleaned text.** Added to record the observable differences the lab asks for (record counts, chunk boundaries, preserved information).

In [7]:
# Step B3 — comparison evidence for fixed vs paragraph-aware chunking (same cleaned text, size=500)
import statistics

def summarise(chunks):
    lens = [len(c) for c in chunks]
    ends_clean = sum(1 for c in chunks if re.search(r"[.!?:;)]\s*$", c.strip()))
    starts_mid = sum(1 for c in chunks if c.strip()[:1].islower())
    starts_heading = sum(1 for c in chunks if re.match(r"\s*(Article \d+|SECTION \d+)", c))
    return {
        "chunks": len(chunks),
        "mean_len": round(statistics.mean(lens), 1),
        "min_len": min(lens),
        "max_len": max(lens),
        "pct_end_at_sentence_or_clause": round(100 * ends_clean / len(chunks), 1),
        "pct_start_mid_sentence": round(100 * starts_mid / len(chunks), 1),
        "chunks_starting_at_heading": starts_heading,
    }

segments = [s.strip() for s in re.split(r"\n\s*\n", clean) if s.strip()]
long_segments = sum(1 for s in segments if len(s) > 500)

COMPARISON = {
    "fixed": summarise(fixed),
    "paragraph-aware": summarise(structured),
    "paragraph_segments": len(segments),
    "paragraph_segments_longer_than_500": long_segments,
}
for k in ("fixed", "paragraph-aware"):
    print(k.ljust(16), COMPARISON[k])
print("Paragraph segments found by blank-line split:", len(segments))
print("...of which longer than 500 characters (fall back to fixed splitting):", long_segments)
print("\nFixed chunk 2 start:", repr(fixed[1][:150]))
print("Paragraph-aware chunk 2 start:", repr(structured[1][:150]))

fixed            {'chunks': 210, 'mean_len': 498.5, 'min_len': 182, 'max_len': 500, 'pct_end_at_sentence_or_clause': 4.8, 'pct_start_mid_sentence': 84.8, 'chunks_starting_at_heading': 0}
paragraph-aware  {'chunks': 213, 'mean_len': 491.2, 'min_len': 74, 'max_len': 500, 'pct_end_at_sentence_or_clause': 5.6, 'pct_start_mid_sentence': 82.2, 'chunks_starting_at_heading': 2}
Paragraph segments found by blank-line split: 14
...of which longer than 500 characters (fall back to fixed splitting): 5

Fixed chunk 2 start: '00042/gid00183/gid00032/gid00031/gid00036/gid00047/gid00042/gid00045\n/gid00014/gid00028/gid00035/gid00052/gid00048/gid00031/gid00031/gid00036/gid00041'
Paragraph-aware chunk 2 start: '00042/gid00183/gid00032/gid00031/gid00036/gid00047/gid00042/gid00045\n/gid00014/gid00028/gid00035/gid00052/gid00048/gid00031/gid00031/gid00036/gid00041'


**Cell 5A2 (CONDITIONAL REQUIRED, after A1), source S01.** Choose one justified chunking configuration and export traceable records. The choice is recorded in `docs/week3_pipeline_decisions.md`.

In [8]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text_S01.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 213
Prepared file: /content/week3_project/data/processed/prepared_text_S01.jsonl
File size: 165241 bytes
Validation: PASS
First record: {'record_id': 'FT-UG-ACADEMIC-REGULATIONS-16th-edition-0001', 'text': 'www.penerbit.utm.my\n2026\n/gid00006/gid00031/gid00036/gid00047/gid00042/gid00045/gid00046\n/gid00014/gid00042/gid00035/gid00031/gid00001/gid00020/gid00035/gid00028/gid00033/gid00045/gid00052/gid00001/gid00014/gid00042/gid00035/gid00031/gid00001/gid00019/gid00028/gid00035/gid00036/gid00040\n/gid00010/gid00041/gid00047/gid00028/gid00041/gid00001/gid00027/gid00028/gid00048/gid00045/gid00028/gid00035/gid00001/gid00014/gid00028/gid00047/gid00001/gid00005/gid00028/gid00045/gid00048/gid00046\n/gid00004/gid', 'source': 'FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf', 'source_sha256': 'd2743dfad87d3be3393732e0416088ad62cf4afb9e6f7355eac47ab715de6a04', 'processing': 'paragraph-aware-char-500',

### Step B3: Validate the prepared result
**B3 basic output contract (REQUIRED).** Run after the chosen pathway. Also inspect representative prepared records, since a non-empty file alone is not proof of correct content.

In [9]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

# Manual content check: two representative prepared records
for r in (exported_records[0], exported_records[len(exported_records) // 2]):
    print("\n", r["record_id"], "| source:", r["source"], "| page:", r["page"], "| processing:", r["processing"])
    print(repr(r["text"][:200]))

Output: /content/week3_project/data/processed/prepared_text_S01.jsonl Size: 165241
Validation: file exists and is non-empty; verify content manually too.

 FT-UG-ACADEMIC-REGULATIONS-16th-edition-0001 | source: FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf | page: None | processing: paragraph-aware-char-500
'www.penerbit.utm.my\n2026\n/gid00006/gid00031/gid00036/gid00047/gid00042/gid00045/gid00046\n/gid00014/gid00042/gid00035/gid00031/gid00001/gid00020/gid00035/gid00028/gid00033/gid00045/gid00052/gid00001/gi'

 FT-UG-ACADEMIC-REGULATIONS-16th-edition-0107 | source: FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf | page: None | processing: paragraph-aware-char-500
'f course meetings (lectures/\npracticals/studio/fieldwork, etc.). If a student is absent from a course \nmeeting, they must inform the course lecturer of the reasons for their \nabsence as soon as possib'


### Step B4: Preserve metadata, lineage and export
**B4.1: register Pathway A.** Uses the selected chunking approach from Cell 5A2.

In [10]:
# Register Pathway A — PDF/TXT (source S01)
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf -> prepared_text_S01.jsonl


In [11]:
# Evidence store (added): keep this source's results for the docs, then move to the next source
try:
    EVIDENCE
except NameError:
    EVIDENCE = {}
EVIDENCE["S01"] = {
    "source_name": f.name, "source_sha256": source_hash(f), "source_bytes": f.stat().st_size,
    "inspection": INSPECTION, "comparison": COMPARISON,
    "chosen_method": CHOSEN_METHOD, "approach": approach, "records_exported": len(records),
    "prepared_name": prepared_path.name, "prepared_sha256": source_hash(prepared_path),
    "prepared_bytes": prepared_path.stat().st_size,
}
print("Evidence stored for S01:", prepared_path.name, "|", len(records), "records")

Evidence stored for S01: prepared_text_S01.jsonl | 213 records


### Source S02 (FAI programme page: Bachelor in Artificial Intelligence with Honours): Step A3, inspect the selected data source (REQUIRED)
Selected source path and type: structured? No. **Unstructured: text (PDF)**. Two representative subsets of this PDF are inspected: a body-text page and a second, differently laid-out page (a table page where one exists). Observed limitations feed the Part B decisions.

In [12]:
# Step A3 — Inspect the selected source (REQUIRED). Added for the lab's inspection evidence.
from pypdf import PdfReader
import re, statistics
from collections import Counter

SOURCE_NAME = 'Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf'
SRC = select_source(relative_path=SOURCE_NAME)
reader = PdfReader(str(SRC))
page_text = [(p.extract_text() or "") for p in reader.pages]
n_pages = len(page_text)
chars = [len(t) for t in page_text]
empty_pages = [i + 1 for i, c in enumerate(chars) if c < 20]

print("Source path:", SRC.relative_to(RAW_DIR))
print("File size (bytes):", SRC.stat().st_size)
print("Pages:", n_pages)
print("PDF metadata title:", (reader.metadata.title if reader.metadata else None))
print("Characters per page: min/median/max =", min(chars), "/", int(statistics.median(chars)), "/", max(chars))
print("Pages with (almost) no extractable text:", empty_pages)

# Subset 1: first page that looks like body text (>= 800 characters)
body_idx = next((i for i, c in enumerate(chars) if c >= 800), 0)
# Subset 2: first page whose text contains a table caption such as "Table 5"
table_idx = next((i for i, t in enumerate(page_text) if re.search(r"^Table \d+", t, re.M)), None)
sub2_label = "Subset 2 (page with a table caption)"
if table_idx is None:
    # No "Table N" caption in this source: use the page with the most lines (list/table-like layout)
    table_idx = max((i for i in range(n_pages) if i != body_idx), key=lambda i: page_text[i].count("\n"))
    sub2_label = "Subset 2 (page with the most lines; no table caption in this source)"

for label, idx in [("Subset 1 (body text page)", body_idx), (sub2_label, table_idx)]:
    if idx is None:
        print("\n" + label + ": none found")
        continue
    t = page_text[idx]
    print("\n" + label + ": PDF page", idx + 1, "| characters:", len(t), "| lines:", t.count("\n") + 1)
    print(repr(t[:400]))

# Layout checks that affect chunking
norm = Counter()
for t in page_text:
    for line in set(l.strip() for l in t.splitlines()):
        if 0 < len(line) < 100:
            norm[re.sub(r"\d+", "#", line)] += 1
repeated = [(l, c) for l, c in norm.most_common(8) if c >= max(3, n_pages // 4)]
hyphen_breaks = sum(len(re.findall(r"\w-\n\w", t)) for t in page_text)
table_captions = sum(len(re.findall(r"^Table \d+", t, re.M)) for t in page_text)
article_headings = sum(len(re.findall(r"^Article \d+", t, re.M)) for t in page_text)
blank_line_breaks = sum(len(re.findall(r"\n\s*\n", t)) for t in page_text)
print("\nRepeated short lines (digits shown as #), appears on N pages:", repeated)
print("Hyphenated line breaks:", hyphen_breaks)
print("Table captions found:", table_captions)
print("'Article N' headings found at line start:", article_headings)
print("Blank-line paragraph breaks across the whole document:", blank_line_breaks)

# Source identity lines (edition / ISBN / publisher) from the first pages
ident = [l.strip() for t in page_text[:4] for l in t.splitlines() if re.search(r"Edition|ISBN|Penerbit|Published", l, re.I)]
print("\nSource identity lines (first pages):", ident[:6])

# Evidence of how a table is extracted
tbl_excerpt = []
if table_idx is not None:
    _lines = page_text[table_idx].splitlines()
    for _j, _l in enumerate(_lines):
        if re.match(r"Table \d+", _l):
            tbl_excerpt = _lines[_j:_j + 8]
            break
    if not tbl_excerpt:
        tbl_excerpt = _lines[:8]   # no table caption: show the first lines of the subset-2 page
print("Table excerpt as extracted:", tbl_excerpt)

INSPECTION = {
    "identity_lines": ident[:6], "table_excerpt": tbl_excerpt,
    "pages": n_pages, "file_bytes": SRC.stat().st_size, "chars_total": sum(chars),
    "chars_min": min(chars), "chars_median": int(statistics.median(chars)), "chars_max": max(chars),
    "empty_pages": empty_pages, "subset1_page": body_idx + 1,
    "subset2_page": (table_idx + 1) if table_idx is not None else None,
    "repeated_lines": repeated, "hyphen_breaks": hyphen_breaks, "table_captions": table_captions,
    "article_headings": article_headings, "blank_line_breaks": blank_line_breaks,
}
assert sum(chars) > 0, "No extractable text: this source may need OCR."

Source path: Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf
File size (bytes): 1372769
Pages: 14
PDF metadata title: None
Characters per page: min/median/max = 828 / 1443 / 3107
Pages with (almost) no extractable text: []

Subset 1 (body text page): PDF page 1 | characters: 2509 | lines: 53
'Copyright © 2026 Faculty of Artificial Intelligence | All Right Reserved.\nBachelor in Artificial Intelligence with Honours\nOverview\nProgramme Educational\nObjectives\nLearning Outcomes\nProgramme Structure\nStudy Plan\nAdmission Criteria\nCareer Prospect\nProgramme Fees\nQuick Facts\nOverview\nBachelor in Artificial Intelligence with honours offers in Faculty of Artificial Intelligence, UTM is a 3-year\nprog'

Subset 2 (page with the most lines; no table caption in this source): PDF page 7 | characters: 2725 | lines: 130
'UNIVERSITI TEKNOLOGI MALAYSIA \nSYARAT KHAS KEMASUKAN MENGIKUT PROGRAM \nBAGI LEPASAN DIPLOMA UA, POLITEKNIK, DVM DAN KELAYAKAN SETARAF \nSESI AKADEMIK

## Part B: Prepare the Data

### Step B1/B2: Pathway A (PDF/TXT) — parsing, cleaning and chunking
**Cell 5A1 (CONDITIONAL REQUIRED), source S02.** Extract text, clean, and compare fixed-size and paragraph-aware chunking.

In [13]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
# Install PDF processing library
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source file
# Project source S02: FAI programme page: Bachelor in Artificial Intelligence with Honours
f = select_source(
    relative_path='Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf'
)

# Step 2: Extract text according to file type
if f.suffix.lower() == ".pdf":
    reader = PdfReader(str(f))
    pages = [
        (i + 1, page.extract_text() or "")
        for i, page in enumerate(reader.pages)
    ]
elif f.suffix.lower() == ".txt":
    pages = [(1, f.read_text(encoding="utf-8"))]
else:
    raise ValueError("Pathway A needs a PDF or TXT source")
# Step 3: Basic text cleaning
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages/segments:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters:", len(clean))
print("Content sample:", repr(clean[:300]))

assert clean, "No usable text: check extraction or OCR needs."
# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]
# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))

        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts
# Step 6: Compare two preparation approaches
fixed = fixed_chunks(clean, size=500)
structured = paragraph_chunks(clean, size=500)

print("\n--- Chunking Comparison ---")
print("Fixed chunks:", len(fixed))
print("Paragraph-aware chunks:", len(structured))

print("\nFixed sample:")
print(repr(fixed[0][:180]))

print("\nParagraph-aware sample:")
print(repr(structured[0][:180]))

print("\nPATHWAY A1 completed successfully.")

Source: Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf
Pages/segments: 14
Extracted characters: 22276
Cleaned characters: 22227
Content sample: 'Copyright © 2026 Faculty of Artificial Intelligence | All Right Reserved.\nBachelor in Artificial Intelligence with Honours\nOverview\nProgramme Educational\nObjectives\nLearning Outcomes\nProgramme Structure\nStudy Plan\nAdmission Criteria\nCareer Prospect\nProgramme Fees\nQuick Facts\nOverview\nBachelor in Art'

--- Chunking Comparison ---
Fixed chunks: 45
Paragraph-aware chunks: 57

Fixed sample:
'Copyright © 2026 Faculty of Artificial Intelligence | All Right Reserved.\nBachelor in Artificial Intelligence with Honours\nOverview\nProgramme Educational\nObjectives\nLearning Outcom'

Paragraph-aware sample:
'Copyright © 2026 Faculty of Artificial Intelligence | All Right Reserved.\nBachelor in Artificial Intelligence with Honours\nOverview\nProgramme Educational\nObjectives\nLearning Outcom'

PATHWAY A1 completed succ

**Step B3 (part 1): compare the two approaches on the same cleaned text.** Added to record the observable differences the lab asks for (record counts, chunk boundaries, preserved information).

In [14]:
# Step B3 — comparison evidence for fixed vs paragraph-aware chunking (same cleaned text, size=500)
import statistics

def summarise(chunks):
    lens = [len(c) for c in chunks]
    ends_clean = sum(1 for c in chunks if re.search(r"[.!?:;)]\s*$", c.strip()))
    starts_mid = sum(1 for c in chunks if c.strip()[:1].islower())
    starts_heading = sum(1 for c in chunks if re.match(r"\s*(Article \d+|SECTION \d+)", c))
    return {
        "chunks": len(chunks),
        "mean_len": round(statistics.mean(lens), 1),
        "min_len": min(lens),
        "max_len": max(lens),
        "pct_end_at_sentence_or_clause": round(100 * ends_clean / len(chunks), 1),
        "pct_start_mid_sentence": round(100 * starts_mid / len(chunks), 1),
        "chunks_starting_at_heading": starts_heading,
    }

segments = [s.strip() for s in re.split(r"\n\s*\n", clean) if s.strip()]
long_segments = sum(1 for s in segments if len(s) > 500)

COMPARISON = {
    "fixed": summarise(fixed),
    "paragraph-aware": summarise(structured),
    "paragraph_segments": len(segments),
    "paragraph_segments_longer_than_500": long_segments,
}
for k in ("fixed", "paragraph-aware"):
    print(k.ljust(16), COMPARISON[k])
print("Paragraph segments found by blank-line split:", len(segments))
print("...of which longer than 500 characters (fall back to fixed splitting):", long_segments)
print("\nFixed chunk 2 start:", repr(fixed[1][:150]))
print("Paragraph-aware chunk 2 start:", repr(structured[1][:150]))

fixed            {'chunks': 45, 'mean_len': 493.9, 'min_len': 227, 'max_len': 500, 'pct_end_at_sentence_or_clause': 4.4, 'pct_start_mid_sentence': 62.2, 'chunks_starting_at_heading': 0}
paragraph-aware  {'chunks': 57, 'mean_len': 386.6, 'min_len': 8, 'max_len': 500, 'pct_end_at_sentence_or_clause': 14.0, 'pct_start_mid_sentence': 35.1, 'chunks_starting_at_heading': 0}
Paragraph segments found by blank-line split: 64
...of which longer than 500 characters (fall back to fixed splitting): 11

Fixed chunk 2 start: 'd\napplications of AI. The program gives students the knowledge and skills to overcome and cope with the\nproblems and opportunities of the emerging fie'
Paragraph-aware chunk 2 start: 'd\napplications of AI. The program gives students the knowledge and skills to overcome and cope with the\nproblems and opportunities of the emerging fie'


**Cell 5A2 (CONDITIONAL REQUIRED, after A1), source S02.** Choose one justified chunking configuration and export traceable records. The choice is recorded in `docs/week3_pipeline_decisions.md`.

In [15]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json
# Step 1: Select the preparation approach
# Change to "fixed" if your Part A1 evidence supports it.
CHOSEN_METHOD = "paragraph-aware"
if CHOSEN_METHOD == "paragraph-aware":
    chosen = structured
    approach = "paragraph-aware-char-500"
elif CHOSEN_METHOD == "fixed":
    chosen = fixed
    approach = "fixed-char-500"
else:
    raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# Step 2: Create traceable records
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": chunk,
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_hash(f),
        "processing": approach,
        "page": None
    }
    for i, chunk in enumerate(chosen, 1)
]
# Page is None because the basic whole-document
# splitter does not preserve page boundaries.
# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text_S02.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )
# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), \
    "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]
assert len(exported_records) == len(records), \
    "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-500
Total chunks exported: 57
Prepared file: /content/week3_project/data/processed/prepared_text_S02.jsonl
File size: 41794 bytes
Validation: PASS
First record: {'record_id': 'Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page-0001', 'text': 'Copyright © 2026 Faculty of Artificial Intelligence | All Right Reserved.\nBachelor in Artificial Intelligence with Honours\nOverview\nProgramme Educational\nObjectives\nLearning Outcomes\nProgramme Structure\nStudy Plan\nAdmission Criteria\nCareer Prospect\nProgramme Fees\nQuick Facts\nOverview\nBachelor in Artificial Intelligence with honours offers in Faculty of Artificial Intelligence, UTM is a 3-year\nprogram and is designed to provide students with a broad background in the principles, methodologies, an', 'source': 'Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf', 'source_sha256': 'b8527fea695e284ed99daf5811ff31f0d7f0adb1c15bf

### Step B3: Validate the prepared result
**B3 basic output contract (REQUIRED).** Run after the chosen pathway. Also inspect representative prepared records, since a non-empty file alone is not proof of correct content.

In [16]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

# Manual content check: two representative prepared records
for r in (exported_records[0], exported_records[len(exported_records) // 2]):
    print("\n", r["record_id"], "| source:", r["source"], "| page:", r["page"], "| processing:", r["processing"])
    print(repr(r["text"][:200]))

Output: /content/week3_project/data/processed/prepared_text_S02.jsonl Size: 41794
Validation: file exists and is non-empty; verify content manually too.

 Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page-0001 | source: Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf | page: None | processing: paragraph-aware-char-500
'Copyright © 2026 Faculty of Artificial Intelligence | All Right Reserved.\nBachelor in Artificial Intelligence with Honours\nOverview\nProgramme Educational\nObjectives\nLearning Outcomes\nProgramme Structu'

 Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page-0029 | source: Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf | page: None | processing: paragraph-aware-char-500
'dan \n Mendapat sekurang-kurangnya kepujian (Gred C) \npada peringkat SPM dalam mata pelajaran berikut: \n • Matematik atau \n• Matematik Tambahan \n dan \n Mendapat sekurang-kurangnya Band 2.0 dalam \nMalay'


### Step B4: Preserve metadata, lineage and export
**B4.1: register Pathway A.** Uses the selected chunking approach from Cell 5A2.

In [17]:
# Register Pathway A — PDF/TXT (source S02)
register_lineage(
    source=f,
    output=prepared_path,
    processing=approach
)

Registered: Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf -> prepared_text_S02.jsonl


In [18]:
# Evidence store (added): keep this source's results for the docs, then move to the next source
try:
    EVIDENCE
except NameError:
    EVIDENCE = {}
EVIDENCE["S02"] = {
    "source_name": f.name, "source_sha256": source_hash(f), "source_bytes": f.stat().st_size,
    "inspection": INSPECTION, "comparison": COMPARISON,
    "chosen_method": CHOSEN_METHOD, "approach": approach, "records_exported": len(records),
    "prepared_name": prepared_path.name, "prepared_sha256": source_hash(prepared_path),
    "prepared_bytes": prepared_path.stat().st_size,
}
print("Evidence stored for S02:", prepared_path.name, "|", len(records), "records")

Evidence stored for S02: prepared_text_S02.jsonl | 57 records


**Cell 6A (REQUIRED).** Generate the common lineage manifest `data/week3_manifest.json`.

In [19]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 2
Saved to: /content/week3_project/data/week3_manifest.json


**Cell 6B (REQUIRED).** Verify the saved manifest.

In [20]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: /content/week3_project/data/week3_manifest.json


## Part C: Build and Document the V1 Data Pipeline

### Step C1: Integrated V1 data pipeline
**Cell 7 (REQUIRED).** Common pipeline utilities. Run once after Part B.

In [21]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


**Cell 8A (CONDITIONAL REQUIRED: PDF/TXT), source S01.** Same source and the same Part B configuration, now as one executable pipeline.

In [22]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Run only if your project uses document data.
# Requires Cell 7 (Common V1 Pipeline Utilities).

!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the source and preparation configuration
# Replace with your actual source and Part B decisions.

SOURCE_A = resolve_source(
    'FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf'
)

CONFIG_A = {
    "chunk_size": 500,
    "method": 'paragraph-aware'
}

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]

    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        pages = [
            page.extract_text() or ""
            for page in reader.pages
        ]
        text = "\n".join(pages)
        page_count = len(pages)

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")
        page_count = None

    else:
        raise ValueError("Expected a PDF or TXT source")

    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )

    return {
        "source": src.name,
        "characters": len(text),
        "pages": page_count
    }


# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]

    # Extract text
    if src.suffix.lower() == ".pdf":
        reader = PdfReader(str(src))
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif src.suffix.lower() == ".txt":
        text = src.read_text(encoding="utf-8")

    else:
        raise ValueError("Expected a PDF or TXT source")

    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()

    size = int(cfg["chunk_size"])
    method = cfg["method"]

    if size <= 0:
        raise ValueError("Chunk size must be positive")

    # Fixed-size chunking
    if method == "fixed":
        chunks = [
            text[i:i + size]
            for i in range(0, len(text), size)
        ]

    # Paragraph-aware chunking
    elif method == "paragraph-aware":
        chunks = []
        current = ""

        for section in re.split(r"\n\s*\n", text):
            section = section.strip()

            if not section:
                continue

            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""

                chunks.extend(
                    section[i:i + size]
                    for i in range(0, len(section), size)
                )

            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += (
                    "\n\n" if current else ""
                ) + section

            else:
                if current:
                    chunks.append(current)

                current = section

        if current:
            chunks.append(current)

    else:
        raise ValueError(
            "Choose 'fixed' or 'paragraph-aware'"
        )

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"

    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {
                "id": i,
                "source": str(src),
                "text": chunk,
                "page": None
            }
            handle.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    # Step 5: Return source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    processing_method = (
        f"{method} text chunks of {size} characters"
    )

    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline
run_A = register_and_run(
    "documents_S01_regulations",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A (S01) completed successfully.")

SUCCESS: documents_S01_regulations | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S01_regulations/pipeline_manifest.json
Cell 8A (S01) completed successfully.


**Cell 8A (second run, source S02).** The same Cell 8A pipeline functions are reused for the second source, with its own source, configuration and `kind`, so its output goes to its own folder.

In [23]:
# Cell 8A — second run for source S02 (uses inspect_documents / prepare_documents defined above)
SOURCE_A2 = resolve_source(
    'Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf'
)

CONFIG_A2 = {
    "chunk_size": 500,
    "method": 'paragraph-aware'
}

run_A2 = register_and_run(
    "documents_S02_programme",
    SOURCE_A2,
    inspect_documents,
    prepare_documents,
    CONFIG_A2
)
print("Cell 8A (S02) completed successfully.")

SUCCESS: documents_S02_programme | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S02_programme/pipeline_manifest.json
Cell 8A (S02) completed successfully.


**Cell 9 (REQUIRED).** Repeatability and provenance checks: re-run the registered pipeline and compare source fingerprints, prepared-output fingerprints and mappings.

In [24]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A-8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents_S01_regulations | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S01_regulations/pipeline_manifest.json
PASS: documents_S01_regulations — source fingerprints, prepared outputs and mappings match
SUCCESS: documents_S02_programme | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S02_programme/pipeline_manifest.json
PASS: documents_S02_programme — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
